# 🧠 BrainCheck — generate and take assessments from your own material or topic

**How this notebook works**
1. Run the cells top to bottom (setup → core → app).
2. Use the interactive panel below the *app* cell. **Nothing is pre-filled: every result is generated from what you upload or type.** Each result shows a *Provenance* box (file name, size, SHA-256 fingerprint) so you can prove it came from your input.
3. Failures never crash the notebook: a bad file or API hiccup gives a readable message and you can simply correct the input and press the button again.
4. No widgets (rare)? Run `run_cli()` in the fallback cell for an `input()`-based version.

## 1. Setup

In [ ]:
# One-time setup (about 1-3 minutes in Colab). Safe to re-run.
import subprocess, sys, shutil
def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    if r.returncode: print("pip warning for", pkgs, "->", r.stderr[-300:])
pip("anthropic", "pypdf", "ipywidgets", "numpy", "scikit-learn", "sentence-transformers", "pytesseract", "pdf2image", "graphviz")
if shutil.which("tesseract") is None or shutil.which("pdftoppm") is None:      # OCR for scanned PDFs
    subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils graphviz", shell=True, capture_output=True)
print("OCR available:", shutil.which("tesseract") is not None, "| setup done")

## 2. Core utilities

In [ ]:
import os, sys, io, re, json, time, math, hashlib, getpass, subprocess, tempfile, html, traceback, difflib, warnings
from pathlib import Path
from dataclasses import dataclass, field
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ config
MAX_FILE_MB = 25                                   # reject bigger uploads
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")
STORE_DIR = os.environ.get("STUDYAI_STORE", "studyai_store")   # point this at Google Drive to persist across sessions

class InputError(Exception):
    """A problem with the user's input that the user can fix (bad file, empty, unsupported...)."""

class AIError(Exception):
    """A problem talking to the AI backend (bad key, rate limit, network...)."""

def sha256(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def esc(s) -> str:
    return html.escape(str(s))

# ------------------------------------------------------------------ AI backend (swap this one function to change provider)
_client = None

def _get_api_key():
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        try:
            from google.colab import userdata          # Colab "Secrets" panel
            key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            key = None
    if not key:
        key = getpass.getpass("Paste your Anthropic API key (input hidden): ").strip()
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
    return key

def reset_api_key():
    """Call this if you pasted a wrong key."""
    global _client
    _client = None
    os.environ.pop("ANTHROPIC_API_KEY", None)

def llm(prompt: str, system: str = "", max_tokens: int = 3000, retries: int = 4) -> str:
    """One text-in/text-out call with retry + back-off for rate limits and transient failures."""
    global _client
    import anthropic
    if _client is None:
        key = _get_api_key()
        if not key:
            raise AIError("No API key provided. Run the API-key cell again.")
        _client = anthropic.Anthropic(api_key=key, max_retries=0)
    delay = 3
    last = None
    for attempt in range(retries + 1):
        try:
            r = _client.messages.create(
                model=MODEL, max_tokens=max_tokens,
                system=system or "You are a careful, accurate study assistant.",
                messages=[{"role": "user", "content": prompt}])
            return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
        except anthropic.AuthenticationError:
            reset_api_key()
            raise AIError("The API key was rejected. Run reset_api_key() and re-enter a valid key.")
        except anthropic.RateLimitError as e:
            last = "rate limit reached"
        except (anthropic.APIConnectionError, anthropic.APITimeoutError) as e:
            last = "network problem"
        except anthropic.APIStatusError as e:
            if e.status_code in (429, 500, 502, 503, 529):
                last = f"temporary server error ({e.status_code})"
            else:
                raise AIError(f"API error {e.status_code}: {getattr(e, 'message', e)}")
        if attempt < retries:
            print(f"   ... {last}; retrying in {delay}s ({attempt + 1}/{retries})")
            time.sleep(delay)
            delay *= 2
    raise AIError(f"AI backend unavailable after {retries + 1} attempts ({last}). Wait a minute and try again.")

def parse_json_loose(text: str):
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    a, b = t.find("{"), t.rfind("}")
    if a != -1 and b > a:
        return json.loads(t[a:b + 1])
    raise ValueError("no JSON object found")

def llm_json(prompt: str, system: str = "", max_tokens: int = 3500) -> dict:
    raw = llm(prompt, system, max_tokens)
    try:
        return parse_json_loose(raw)
    except Exception:
        fixed = llm("The following was supposed to be a single valid JSON object but is malformed. "
                    "Return ONLY the corrected JSON object, nothing else.\n\n" + raw, "You repair JSON.", max_tokens)
        try:
            return parse_json_loose(fixed)
        except Exception as e:
            raise AIError(f"The AI returned a response that could not be parsed ({e}). Try again.")

# ------------------------------------------------------------------ validation
DOC_EXT = {".pdf", ".txt"}
AUDIO_EXT = {".mp3", ".wav", ".m4a"}

def validate_upload(name: str, data: bytes, allowed: set, max_mb: float = None) -> str:
    """Return the lower-case extension, or raise InputError with a fixable message."""
    max_mb = MAX_FILE_MB if max_mb is None else max_mb
    ext = Path(name).suffix.lower()
    if ext not in allowed:
        raise InputError(f"'{name}': unsupported format '{ext or 'none'}'. Allowed: {', '.join(sorted(allowed))}")
    if data is None or len(data) == 0:
        raise InputError(f"'{name}' is empty (0 bytes). Choose a different file.")
    if len(data) > max_mb * 1024 * 1024:
        raise InputError(f"'{name}' is {len(data) / 1048576:.1f} MB; the limit is {max_mb} MB. Split or compress it.")
    if ext == ".pdf" and b"%PDF" not in data[:1024]:
        raise InputError(f"'{name}' does not look like a real PDF (wrong header). It may be a renamed file.")
    if ext == ".wav" and not (data[:4] == b"RIFF" and data[8:12] == b"WAVE"):
        raise InputError(f"'{name}' does not look like a real WAV file (wrong header).")
    return ext

# ------------------------------------------------------------------ small text helpers
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(s).lower())).strip()

def words(s: str) -> list:
    return re.findall(r"\w+", str(s).lower())

def jaccard(a: str, b: str) -> float:
    A, B = set(words(a)), set(words(b))
    return len(A & B) / max(1, len(A | B))

def split_text(t: str, size: int = 900, overlap: int = 150) -> list:
    t = t.strip()
    out, i = [], 0
    while i < len(t):
        j = min(len(t), i + size)
        if j < len(t):
            k = max(t.rfind(". ", i + size // 2, j), t.rfind("\n", i + size // 2, j))
            if k != -1:
                j = k + 1
        out.append(t[i:j].strip())
        if j >= len(t):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if len(c) >= 30 or len(out) == 1]

# ------------------------------------------------------------------ notebook UI helpers
try:
    import ipywidgets as W
    from IPython.display import display, clear_output, HTML, Markdown
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    try:
        from IPython.display import display, clear_output, HTML, Markdown
    except Exception:
        display = print
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()     # lets ipywidgets (incl. FileUpload) work in Colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CSS = """<style>
.sn-card{border:1px solid #8884;border-radius:10px;padding:12px 16px;margin:8px 0}
.sn-ok{background:#2e7d3218;border-left:4px solid #2e7d32}.sn-warn{background:#ef6c0018;border-left:4px solid #ef6c00}
.sn-bad{background:#c6282818;border-left:4px solid #c62828}.sn-info{background:#1565c018;border-left:4px solid #1565c0}
.sn-mark{background:#ffeb3b88;padding:0 2px}.sn-unc{background:#ff980055;border-bottom:2px dotted #e65100;padding:0 2px}
.sn-small{font-size:.85em;opacity:.8}.sn-table td,.sn-table th{padding:4px 10px;border-bottom:1px solid #8883;text-align:left;vertical-align:top}
</style>"""

def show_html(s: str):
    display(HTML(CSS + s))

def callout(kind: str, msg: str) -> str:
    icon = {"ok": "✅", "warn": "⚠️", "bad": "⛔", "info": "ℹ️"}[kind]
    return f"<div class='sn-card sn-{kind}'>{icon} {msg}</div>"

def uploaded_items(up) -> list:
    """Normalise ipywidgets FileUpload .value across v7 (dict) and v8 (tuple) -> [(name, bytes)]."""
    v = up.value
    items = []
    if isinstance(v, dict):
        for name, meta in v.items():
            items.append((name, bytes(meta["content"])))
    else:
        for f in v:
            items.append((f["name"], bytes(f["content"])))
    return items

def reset_upload(up):
    for fn in (lambda: up.value.clear(), lambda: setattr(up, "value", ()), lambda: setattr(up, "_counter", 0)):
        try:
            fn()
        except Exception:
            pass

def cli_upload(allowed: set) -> list:
    """Fallback when ipywidgets is unavailable: Colab file picker, else typed paths."""
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        return [(n, d) for n, d in up.items()]
    raw = input(f"Path(s) to file(s) ({', '.join(sorted(allowed))}), separated by commas: ")
    out = []
    for p in [x.strip() for x in raw.split(",") if x.strip()]:
        try:
            out.append((Path(p).name, Path(p).read_bytes()))
        except Exception as e:
            print(f"⚠️ Cannot read '{p}': {e}")
    return out

def friendly(fn, *a, **k):
    """Run fn; print a readable message instead of crashing the notebook."""
    try:
        return fn(*a, **k)
    except InputError as e:
        show_html(callout("warn", f"<b>Input problem:</b> {esc(e)}<br><span class='sn-small'>Fix the input and try again — nothing else was affected.</span>"))
    except AIError as e:
        show_html(callout("bad", f"<b>AI backend:</b> {esc(e)}<br><span class='sn-small'>Your uploaded data is still loaded; just press the button again.</span>"))
    except Exception as e:
        show_html(callout("bad", f"<b>Unexpected error:</b> {esc(type(e).__name__)}: {esc(e)}<br><span class='sn-small'>The notebook is still usable. Details below.</span>"))
        traceback.print_exc(limit=3)
    return None

In [ ]:
import numpy as np

@dataclass
class LoadedDoc:
    doc_id: str
    name: str
    sha: str
    size: int
    kind: str                      # "pdf" or "txt"
    pages: list                    # [(page_or_section_no, text)]
    student: str = ""
    notes: list = field(default_factory=list)
    ocr_pages: list = field(default_factory=list)

    @property
    def text(self) -> str:
        return "\n\n".join(t for _, t in self.pages)

    @property
    def text_with_pages(self) -> str:
        lab = "Page" if self.kind == "pdf" else "Section"
        return "\n\n".join(f"[{lab} {n}]\n{t}" for n, t in self.pages)

    @property
    def label(self) -> str:
        return "p." if self.kind == "pdf" else "§"

def clean_text(t: str) -> str:
    t = (t or "").replace("\x00", "")
    t = re.sub(r"-\n(\w)", r"\1", t)
    t = re.sub(r"[ \t\f\v]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def _ocr_pages(data: bytes, page_numbers: list):
    try:
        import pytesseract
        from pdf2image import convert_from_bytes
        pytesseract.get_tesseract_version()
    except Exception:
        return {}, "OCR is not installed (run the install cell, which tries to apt-get install tesseract-ocr + poppler-utils)."
    out, err = {}, None
    for n in page_numbers[:80]:
        try:
            img = convert_from_bytes(data, dpi=200, first_page=n, last_page=n)[0]
            out[n] = pytesseract.image_to_string(img)
        except Exception as e:
            err = f"OCR failed on page {n}: {e}"
    return out, err

def extract_pdf_pages(data: bytes, name: str, use_ocr: bool = True):
    try:
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(data))
        if reader.is_encrypted:
            try:
                ok = reader.decrypt("")
            except Exception:
                ok = 0
            if not ok:
                raise InputError(f"'{name}' is password-protected. Remove the password and upload again.")
        raw = []
        for i, p in enumerate(reader.pages, 1):
            try:
                raw.append((i, clean_text(p.extract_text() or "")))
            except Exception:
                raw.append((i, ""))
    except InputError:
        raise
    except Exception as e:
        raise InputError(f"'{name}' could not be read as a PDF (corrupt or damaged): {e}")
    if not raw:
        raise InputError(f"'{name}' has no pages.")
    notes, ocr_done = [], []
    blank = [n for n, t in raw if len(t) < 25]
    if blank and use_ocr:
        res, err = _ocr_pages(data, blank)
        d = dict(raw)
        for n, t in res.items():
            t = clean_text(t)
            if len(t) >= 25:
                d[n] = t
                ocr_done.append(n)
        raw = sorted(d.items())
        if ocr_done:
            notes.append(f"OCR was used on {len(ocr_done)} page(s): {ocr_done[:15]}{'...' if len(ocr_done) > 15 else ''} — OCR text may contain errors.")
        if err and len(res) < len(blank):
            notes.append(err)
    still_blank = [n for n, t in raw if len(t) < 25]
    if len(still_blank) == len(raw):
        raise InputError(f"'{name}' looks like a scanned/image-only PDF and no text could be extracted"
                         f"{' (OCR unavailable or found nothing)' if use_ocr else ' (OCR is switched off)'}. "
                         "Enable OCR / install tesseract, or upload a text-based PDF.")
    if still_blank:
        notes.append(f"{len(still_blank)} page(s) had no readable text and were skipped: {still_blank[:15]}")
    pages = [(n, t) for n, t in raw if len(t) >= 25]
    return pages, ocr_done, notes

def decode_text(data: bytes, name: str) -> str:
    for enc in ("utf-8-sig", "utf-16", "cp1252", "latin-1"):
        try:
            t = data.decode(enc)
            if t.count("\x00") > len(t) * 0.01:
                continue
            return t
        except Exception:
            continue
    raise InputError(f"'{name}' could not be decoded as text.")

def load_document(name: str, data: bytes, student: str = "", use_ocr: bool = True) -> LoadedDoc:
    ext = validate_upload(name, data, DOC_EXT)
    sha = sha256(data)
    if ext == ".pdf":
        pages, ocr_done, notes = extract_pdf_pages(data, name, use_ocr)
        kind = "pdf"
    else:
        t = clean_text(decode_text(data, name))
        if len(t) < 25:
            raise InputError(f"'{name}' contains almost no text ({len(t)} characters).")
        parts = split_text(t, 3000, 0) or [t]
        pages, ocr_done, notes, kind = list(enumerate(parts, 1)), [], [], "txt"
    total = sum(len(t) for _, t in pages)
    if total < 100:
        raise InputError(f"'{name}' has too little text ({total} characters) to work with.")
    return LoadedDoc(doc_id=sha[:10], name=name, sha=sha, size=len(data), kind=kind, pages=pages,
                     student=(student or "").strip(), notes=notes, ocr_pages=ocr_done)

# ------------------------------------------------------------------ persistence ("previously uploaded documents")
def save_doc(doc: LoadedDoc):
    try:
        Path(STORE_DIR).mkdir(parents=True, exist_ok=True)
        Path(STORE_DIR, f"{doc.doc_id}.json").write_text(json.dumps(doc.__dict__, ensure_ascii=False))
    except Exception as e:
        print(f"(could not save '{doc.name}' to {STORE_DIR}: {e})")

def load_saved_docs() -> list:
    out = []
    for p in sorted(Path(STORE_DIR).glob("*.json")) if Path(STORE_DIR).exists() else []:
        try:
            d = json.loads(p.read_text())
            d["pages"] = [tuple(x) for x in d["pages"]]
            out.append(LoadedDoc(**d))
        except Exception:
            pass
    return out

# ------------------------------------------------------------------ chunking + retrieval
def chunk_doc(doc: LoadedDoc, size: int = 900, overlap: int = 150) -> list:
    chunks = []
    for page, text in doc.pages:
        for j, c in enumerate(split_text(text, size, overlap)):
            chunks.append(dict(doc_id=doc.doc_id, doc_name=doc.name, student=doc.student,
                               page=page, label=doc.label, text=c))
    return chunks

class Index:
    """Dense retrieval (sentence-transformers) with an automatic TF-IDF fallback."""
    def __init__(self):
        self.chunks, self._E, self._tf, self._M = [], None, None, None
        try:
            from sentence_transformers import SentenceTransformer
            self._st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
            self.backend = "sentence-transformers (dense)"
        except Exception as e:
            self._st, self.backend = None, "TF-IDF (fallback)"
        self.min_relevance = 0.30 if self._st is not None else 0.10     # below this, passages are "not relevant"
        self.support_score = 0.45 if self._st is not None else 0.20

    def has_doc(self, doc_id): return any(c["doc_id"] == doc_id for c in self.chunks)

    def add(self, doc: LoadedDoc):
        if self.has_doc(doc.doc_id):
            return
        new = chunk_doc(doc)
        if self._st is not None and new:
            E = self._st.encode([c["text"] for c in new], normalize_embeddings=True, show_progress_bar=False)
            for c, e in zip(new, E):
                c["emb"] = e
        self.chunks += new
        self._refit()

    def remove(self, doc_id):
        self.chunks = [c for c in self.chunks if c["doc_id"] != doc_id]
        self._refit()

    def _refit(self):
        if not self.chunks:
            self._E = self._M = self._tf = None
            return
        if self._st is not None:
            self._E = np.vstack([c["emb"] for c in self.chunks])
        else:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self._tf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
            self._M = self._tf.fit_transform([c["text"] for c in self.chunks])

    def _scores(self, q):
        if self._st is not None:
            return self._E @ self._st.encode([q], normalize_embeddings=True, show_progress_bar=False)[0]
        return (self._M @ self._tf.transform([q]).T).toarray().ravel()

    def search(self, query, k=5, doc_ids=None, student=None):
        if not self.chunks or not str(query).strip():
            return []
        s = self._scores(query)
        order = np.argsort(-s)
        out = []
        for i in order:
            c = self.chunks[i]
            if doc_ids and c["doc_id"] not in doc_ids:
                continue
            if student and student.strip().lower() not in ("", "all students") and c["student"].lower() != student.strip().lower():
                continue
            r = {k_: v for k_, v in c.items() if k_ != "emb"}
            r["score"] = float(s[i])
            out.append(r)
            if len(out) >= k:
                break
        return out

In [ ]:
# Enter your Anthropic API key once. Order tried: environment variable -> Colab Secrets (ANTHROPIC_API_KEY) -> hidden prompt.
# The AI backend is the single function llm() in the next cell — swap it to use another provider.
_get_api_key()
print("Key loaded. Model:", MODEL)

## 3. BrainCheck app
1. Add study material (PDF/TXT upload) and/or select documents already uploaded (documents saved by QuickNotes/AskAway into the same `studyai_store` folder also appear). You can also type a **custom topic** — with documents selected it focuses the questions; without documents it generates from general knowledge (clearly labelled).
2. Pick number of questions, **difficulty** (Easy/Medium/Hard) and **type** (MCQ, True/False, Fill in the Blanks, Short Answer, Descriptive) → **Generate assessment**.
3. Answer in the boxes → **Submit answers** for scores, explanations, missing concepts, misconceptions and improvement tips.
4. **Retake** generates brand-new questions (earlier ones are excluded).

MCQ/True-False are graded exactly; fill-in, short and descriptive answers are graded by meaning against the expected answer, key concepts and the source quote — not by keyword matching.

In [ ]:
import random

DOCS = {}                      # doc_id -> LoadedDoc  (previously uploaded + saved documents appear here)
INDEX = Index()
for _d in load_saved_docs():   # documents saved by QuickNotes/AskAway in the same runtime/Drive folder
    DOCS[_d.doc_id] = _d; INDEX.add(_d)
print(f"Retrieval backend: {INDEX.backend} · {len(DOCS)} previously saved document(s) found in '{STORE_DIR}'")

QTYPES = {"MCQ": "mcq", "True/False": "tf", "Fill in the Blanks": "fill", "Short Answer": "short", "Descriptive": "desc"}
DIFF_GUIDE = {"Easy": "recall of basic facts and definitions", "Medium": "understanding and application of concepts",
              "Hard": "multi-step reasoning, comparison, analysis or edge cases"}
STATE = {"questions": [], "settings": None, "previous": [], "last_eval": None}

BC_SYSTEM = ("You are an expert examiner. Questions and expected answers must be accurate and, when source passages are given, "
             "answerable from them. Grade students fairly on meaning, not wording.")

# ------------------------------------------------------------------ source selection
def gather_source(doc_ids, topic, max_chars=11000) -> (str, list):
    """Passages from the selected documents; focused on the topic if one is given, otherwise spread evenly."""
    if not doc_ids:
        return "", []
    if topic.strip():
        hits = INDEX.search(topic, k=14, doc_ids=set(doc_ids))
        hits = [h for h in hits if h["score"] >= INDEX.min_relevance * 0.8] or hits[:4]
    else:
        allc = [c for c in INDEX.chunks if c["doc_id"] in doc_ids]
        step = max(1, len(allc) // 14)
        hits = [dict(c, score=1.0) for c in allc[::step]][:14]
    text, used, total = [], [], 0
    for h in hits:
        if total + len(h["text"]) > max_chars:
            break
        text.append(f"[{h['doc_name']} {h['label']}{h['page']}]\n{h['text']}"); used.append(h); total += len(h["text"])
    return "\n\n".join(text), used

def generate_questions(doc_ids, topic, n, difficulty, qtype_label) -> list:
    qt = QTYPES[qtype_label]
    if not doc_ids and not topic.strip():
        raise InputError("Choose at least one document OR type a topic.")
    src, used = gather_source(doc_ids, topic)
    sourced = bool(src)
    shape = {"mcq": '"options": ["4 distinct options"], "answer": "the exact text of the correct option"',
             "tf": '"options": ["True","False"], "answer": "True" or "False"',
             "fill": '"options": [], "answer": "the missing word/phrase (question contains ____)"',
             "short": '"options": [], "answer": "a 1-3 sentence model answer"',
             "desc": '"options": [], "answer": "a complete model answer (5-8 sentences)"'}[qt]
    avoid = "\n".join("- " + q for q in STATE["previous"][-30:])
    prompt = (f"{'SOURCE PASSAGES:' + chr(10) + src if sourced else 'No source documents: use well-established general knowledge of the topic.'}\n\n"
              f"TOPIC: {topic or '(whole source material)'}\nCreate {n} {qtype_label} questions at {difficulty} difficulty ({DIFF_GUIDE[difficulty]}).\n"
              f"{'Questions MUST be answerable from the source passages only.' if sourced else ''}\n"
              f"{'Do NOT repeat or paraphrase these earlier questions:' + chr(10) + avoid if avoid else ''}\n\n"
              'Return JSON: {"questions": [{"question": "...", ' + shape + ', "explanation": "why the answer is right", '
              '"key_concepts": ["concepts needed for a full answer"], "source_quote": "exact supporting quote from the source (empty if none)"}]}')
    out = llm_json(prompt, BC_SYSTEM, 5000)
    qs = out.get("questions", [])
    if not qs:
        raise AIError("The AI returned no questions. Try again or change the topic.")
    clean = []
    for q in qs[:n]:
        q["type"] = qt
        q["sourced"] = sourced
        if qt == "mcq":
            opts = [str(o) for o in q.get("options", [])]
            ans = str(q.get("answer", "")).strip()
            if ans not in opts:
                m = difflib.get_close_matches(ans, opts, 1, 0.6)
                if m: ans = m[0]
                else: opts.append(ans)
            random.shuffle(opts)
            q["options"], q["answer"] = opts, ans
        elif qt == "tf":
            q["options"] = ["True", "False"]
            q["answer"] = "True" if str(q.get("answer", "")).strip().lower().startswith("t") else "False"
        else:
            q["options"] = []
        clean.append(q)
    STATE["previous"] += [q["question"] for q in clean]
    return clean

# ------------------------------------------------------------------ evaluation
def evaluate(questions, answers) -> dict:
    results = [None] * len(questions)
    to_judge = []
    for i, (q, a) in enumerate(zip(questions, answers)):
        a = (a or "").strip()
        if not a:
            results[i] = dict(score=0.0, verdict="unanswered", missing=q.get("key_concepts", []), misconceptions=[], feedback="No answer given.")
        elif q["type"] in ("mcq", "tf"):
            ok = norm(a) == norm(q["answer"])
            results[i] = dict(score=1.0 if ok else 0.0, verdict="correct" if ok else "incorrect", missing=[], misconceptions=[], feedback="")
            if not ok:
                to_judge.append(i)       # ask the examiner only to EXPLAIN why the chosen option is wrong
        elif q["type"] == "fill" and norm(a) == norm(q["answer"]):
            results[i] = dict(score=1.0, verdict="correct", missing=[], misconceptions=[], feedback="Correct.")
        else:
            to_judge.append(i)
    for b in range(0, len(to_judge), 6):
        batch = to_judge[b:b + 6]
        items = []
        for i in batch:
            q = questions[i]
            items.append(dict(id=i, type=q["type"], question=q["question"], expected_answer=q["answer"],
                              key_concepts=q.get("key_concepts", []), source_quote=q.get("source_quote", ""),
                              student_answer=answers[i],
                              objectively_graded=("incorrect" if q["type"] in ("mcq", "tf") else "needs your grading")))
        out = llm_json("Grade these student answers against the expected answers and key concepts. Judge MEANING: accept correct paraphrases, "
                       "synonyms and equivalent formulations; penalise missing key concepts, wrong facts and contradictions. "
                       "For 'objectively_graded=incorrect' items (mcq/tf) the score is already 0 — only explain what misconception the chosen option reveals.\n\n"
                       + json.dumps(items, ensure_ascii=False) +
                       '\n\nReturn JSON: {"grades": [{"id": 0, "score": 0.0-1.0, "verdict": "correct|partially correct|incorrect", '
                       '"missing_concepts": ["..."], "misconceptions": ["specific wrong ideas in the answer"], "feedback": "1-3 sentences telling the student what to fix"}]}',
                       BC_SYSTEM, 3500)
        by_id = {int(g.get("id", -1)): g for g in out.get("grades", [])}
        for i in batch:
            g = by_id.get(i)
            q = questions[i]
            if g is None:
                results[i] = dict(score=0.0, verdict="not graded", missing=[], misconceptions=[], feedback="The AI did not return a grade for this answer — press Submit again.")
                continue
            sc = max(0.0, min(1.0, float(g.get("score", 0))))
            if q["type"] in ("mcq", "tf"):
                sc = 0.0
            results[i] = dict(score=sc, verdict=g.get("verdict", ""), missing=g.get("missing_concepts", []),
                              misconceptions=g.get("misconceptions", []), feedback=g.get("feedback", ""))
    total = sum(r["score"] for r in results)
    weak = [c for r in results for c in r["missing"]]
    summary = ""
    if any(r['verdict'] != 'unanswered' for r in results) and total < len(results):
        try:
            summary = llm("A student just took a quiz. Based on these results write 3-5 concrete study suggestions (bullets, no preamble).\n\n"
                          + json.dumps([dict(q=q["question"], score=r["score"], missing=r["missing"], misconceptions=r["misconceptions"])
                                        for q, r in zip(questions, results)], ensure_ascii=False), BC_SYSTEM, 700)
        except AIError:
            summary = "(improvement suggestions unavailable right now — see per-question feedback)"
    return dict(results=results, total=total, max=len(results), weak=weak, advice=summary)

def render_questions_html(qs):
    s = STATE["settings"]
    return callout("info", f"<b>{len(qs)}</b> {esc(s['type'])} question(s) · {esc(s['difficulty'])} · "
                           f"{'from your documents' + (' / topic: ' + esc(s['topic']) if s['topic'] else '') if qs and qs[0]['sourced'] else 'topic only — generated from general knowledge, not from your notes'}")

def render_eval(ev):
    pct = 100 * ev["total"] / max(1, ev["max"])
    kind = "ok" if pct >= 70 else ("warn" if pct >= 40 else "bad")
    h = [callout(kind, f"<h3 style='margin:0'>Score: {ev['total']:.1f} / {ev['max']} ({pct:.0f}%)</h3>")]
    for i, (q, r) in enumerate(zip(STATE["questions"], ev["results"]), 1):
        k = "ok" if r["score"] >= 0.99 else ("warn" if r["score"] > 0 else "bad")
        h.append(f"<div class='sn-card sn-{k}'><b>Q{i}. {esc(q['question'])}</b> <span class='sn-small'>— {r['score']:.1f}/1 · {esc(r['verdict'])}</span><br>"
                 f"<b>Your answer:</b> {esc(STATE['answers'][i - 1] or '—')}<br><b>Expected:</b> {esc(q['answer'])}<br>"
                 f"<b>Why:</b> {esc(q.get('explanation', ''))}")
        if r["feedback"]: h.append(f"<br><b>Feedback:</b> {esc(r['feedback'])}")
        if r["missing"]: h.append("<br><b>Missing concepts:</b> " + esc("; ".join(map(str, r["missing"]))))
        if r["misconceptions"]: h.append("<br><b>Misconceptions:</b> " + esc("; ".join(map(str, r["misconceptions"]))))
        if q.get("source_quote"): h.append(f"<br><span class='sn-small'>📖 source: “{esc(q['source_quote'])}”</span>")
        h.append("</div>")
    if ev["advice"]:
        h.append(f"<div class='sn-card sn-info'><b>How to improve</b><br><div style='white-space:pre-wrap'>{esc(ev['advice'])}</div></div>")
    show_html("".join(h))

def ingest(items, use_ocr=True):
    added = []
    for name, data in items:
        try:
            d = load_document(name, data, "", use_ocr)
            if d.doc_id not in DOCS:
                DOCS[d.doc_id] = d; INDEX.add(d); save_doc(d)
            added.append(d)
        except InputError as e:
            show_html(callout("warn", f"<b>Skipped:</b> {esc(e)}"))
        except Exception as e:
            show_html(callout("bad", f"<b>Skipped '{esc(name)}':</b> {esc(e)}"))
    return added

# ------------------------------------------------------------------ widgets
if HAS_WIDGETS:
    up = W.FileUpload(accept=".pdf,.txt", multiple=True, description="Upload material")
    ocr_cb = W.Checkbox(value=True, description="OCR scanned pages", indent=False)
    b_up = W.Button(description="Add uploaded files", icon="plus")
    sel_docs = W.SelectMultiple(options=[], rows=5, description="Documents:", layout=W.Layout(width="520px"))
    topic = W.Text(placeholder="optional custom topic, e.g. 'TCP congestion control'", description="Topic:", layout=W.Layout(width="520px"))
    n_q = W.IntSlider(value=5, min=1, max=15, description="Questions:")
    dd_diff = W.Dropdown(options=list(DIFF_GUIDE), value="Medium", description="Difficulty:")
    dd_type = W.Dropdown(options=list(QTYPES), value="MCQ", description="Type:")
    b_gen = W.Button(description="Generate assessment", button_style="primary", icon="magic")
    b_retake = W.Button(description="Retake (new questions)", icon="refresh", disabled=True)
    b_submit = W.Button(description="Submit answers", button_style="success", icon="check", disabled=True)
    b_fill = W.Button(description="Test helper: fill with expected answers", icon="flask", disabled=True)
    lib_out, q_box, out = W.Output(), W.VBox([]), W.Output()
    ans_widgets = []

    def refresh_docs():
        sel_docs.options = [(f"{d.name} ({len(d.text):,} chars)", d.doc_id) for d in DOCS.values()]
        with lib_out:
            clear_output(); print(f"{len(DOCS)} document(s) available.")

    def on_up(_):
        items = uploaded_items(up)
        with lib_out:
            if not items:
                show_html(callout("warn", "Choose file(s) first."))
                return
        added = friendly(ingest, items, ocr_cb.value) or []
        refresh_docs()
        if added:
            sel_docs.value = tuple(set(sel_docs.value) | {d.doc_id for d in added})
        reset_upload(up)

    def build_answer_widgets(qs):
        ans_widgets.clear(); rows = []
        for i, q in enumerate(qs, 1):
            if q["type"] in ("mcq", "tf"):
                w = W.RadioButtons(options=q["options"], value=None, layout=W.Layout(width="95%"))
            elif q["type"] == "fill":
                w = W.Text(placeholder="your answer", layout=W.Layout(width="60%"))
            else:
                w = W.Textarea(placeholder="your answer", layout=W.Layout(width="95%", height="60px" if q["type"] == "short" else "150px"))
            ans_widgets.append(w)
            rows.append(W.VBox([W.HTML(f"<b>Q{i}.</b> {esc(q['question'])}"), w]))
        q_box.children = rows

    def on_gen(_):
        with out:
            clear_output()
            def run():
                STATE["settings"] = dict(topic=topic.value.strip(), difficulty=dd_diff.value, type=dd_type.value)
                print("Generating questions ...")
                qs = generate_questions(list(sel_docs.value), topic.value, n_q.value, dd_diff.value, dd_type.value)
                STATE["questions"], STATE["last_eval"] = qs, None
                clear_output(); show_html(render_questions_html(qs))
                build_answer_widgets(qs)
                for b in (b_submit, b_fill, b_retake): b.disabled = False
            friendly(run)

    def on_submit(_):
        with out:
            clear_output()
            def run():
                ans = [(w.value or "").strip() if not isinstance(w.value, tuple) else "" for w in ans_widgets]
                STATE["answers"] = ans
                print("Evaluating ...")
                ev = evaluate(STATE["questions"], ans)
                STATE["last_eval"] = ev
                clear_output(); render_eval(ev)
            friendly(run)

    def on_fill(_):
        for w, q in zip(ans_widgets, STATE["questions"]):
            w.value = q["answer"]
        with out:
            clear_output(); print("Filled with the generated expected answers (to test the 'correct answer' path). Edit some of them to test wrong answers, then Submit.")

    b_up.on_click(on_up); b_gen.on_click(on_gen); b_retake.on_click(on_gen); b_submit.on_click(on_submit); b_fill.on_click(on_fill)
    refresh_docs()
    display(W.VBox([W.HTML("<h3>🧠 BrainCheck — build and take an assessment from your own material</h3>"),
                    W.HBox([up, b_up, ocr_cb]), lib_out, sel_docs, topic,
                    W.HBox([n_q, dd_diff, dd_type]), W.HBox([b_gen, b_retake]),
                    q_box, W.HBox([b_submit, b_fill]), out]))
else:
    print("ipywidgets not available — use run_cli().")

def run_cli():
    """Plain input()-based alternative."""
    if not DOCS and input("Upload documents? [y/N] ").lower().startswith("y"):
        friendly(ingest, cli_upload(DOC_EXT))
    ids = []
    if DOCS:
        for i, d in enumerate(DOCS.values()): print(i, d.name)
        raw = input("Document numbers to use (comma separated, blank = none): ")
        ids = [list(DOCS)[int(x)] for x in raw.split(",") if x.strip().isdigit()]
    tp = input("Custom topic (optional): ")
    n = int(input("How many questions [5]: ") or 5)
    df = input(f"Difficulty {list(DIFF_GUIDE)} [Medium]: ") or "Medium"
    ty = input(f"Type {list(QTYPES)} [MCQ]: ") or "MCQ"
    STATE["settings"] = dict(topic=tp, difficulty=df, type=ty)
    qs = friendly(generate_questions, ids, tp, n, df, ty)
    if not qs: return
    STATE["questions"] = qs; ans = []
    for i, q in enumerate(qs, 1):
        print(f"\nQ{i}. {q['question']}")
        for j, o in enumerate(q["options"]): print(f"   {j + 1}) {o}")
        a = input("Your answer: ").strip()
        ans.append(q["options"][int(a) - 1] if q["options"] and a.isdigit() and 0 < int(a) <= len(q["options"]) else a)
    STATE["answers"] = ans
    ev = friendly(evaluate, qs, ans)
    if ev: render_eval(ev)

### Fallback (only if widgets don't show)
`run_cli()` runs the whole flow with plain `input()`.

In [ ]:
# run_cli()

## Custom Input Testing and Validation

This section lets you verify, with your own inputs, that the notebook behaves as required. The automated cell tests the *input guards* using files created on the fly; the manual table lists the behaviours to check with your real files.

### Manual acceptance tests
| # | Do this | You should see |
|---|---|---|
| 1 | Upload material, select it, *Generate* | Questions answerable from **your** text; each result shows its source quote after grading |
| 2 | Type a custom topic only (no documents) | Questions on the topic, labelled "generated from general knowledge, not from your notes" |
| 3 | Change count / difficulty / type and regenerate | The quiz changes accordingly (Hard = reasoning-style, MCQ = 4 options, etc.) |
| 4 | Click **Test helper: fill with expected answers** → Submit | Full marks — the "correct understanding" path |
| 5 | Edit some answers to be wrong or incomplete (e.g. keep only half the idea) → Submit | Lower scores with **missing concepts**, **misconceptions** and feedback |
| 6 | Submit with blanks | Those questions score 0 as *unanswered*; the notebook doesn't crash |
| 7 | Press **Retake** | New questions on the same material |
| 8 | Select 2+ documents, then 1 | Questions follow only the selected documents |

In [ ]:
def _doc_guard_tests():
    """Checks the input validators with files built on the fly (these test the *guards*, not the outputs)."""
    from pypdf import PdfWriter
    results = []
    def expect_error(label, fn):
        try:
            fn()
            results.append((label, "❌ accepted (should have been rejected)"))
        except InputError as e:
            results.append((label, f"✅ rejected: {str(e)[:110]}"))
        except Exception as e:
            results.append((label, f"❌ crashed with {type(e).__name__}: {e}"))
    blank = io.BytesIO(); w = PdfWriter(); w.add_blank_page(300, 300); w.write(blank)
    expect_error("empty .txt file", lambda: load_document("empty.txt", b""))
    expect_error("whitespace-only .txt", lambda: load_document("space.txt", b"   \n  \n"))
    expect_error("empty .pdf file", lambda: load_document("empty.pdf", b""))
    expect_error("unsupported format (.docx)", lambda: load_document("notes.docx", b"PK\x03\x04 something"))
    expect_error("renamed/corrupt PDF", lambda: load_document("fake.pdf", b"this is not a pdf at all" * 20))
    expect_error("truncated PDF", lambda: load_document("cut.pdf", b"%PDF-1.4\n1 0 obj\n<<"))
    expect_error("oversized file (limit forced to 0.001 MB)", lambda: validate_upload("big.txt", b"x" * 5000, DOC_EXT, max_mb=0.001))
    expect_error("image-only / scanned-style PDF (blank page, OCR on)", lambda: load_document("scan.pdf", blank.getvalue(), use_ocr=True))
    try:
        d = load_document("ok.txt", ("Photosynthesis converts light energy into chemical energy. " * 6).encode())
        results.append(("valid .txt", f"✅ accepted ({len(d.text)} chars)"))
    except Exception as e:
        results.append(("valid .txt", f"❌ {e}"))
    rows = "".join(f"<tr><td>{esc(a)}</td><td>{esc(b)}</td></tr>" for a, b in results)
    show_html(f"<table class='sn-table'><tr><th>Test</th><th>Result</th></tr>{rows}</table>")
    return all(r[1].startswith("✅") for r in results)




def grading_logic_test():
    """Offline unit test of the objective grading paths (no AI call needed): correct, blank and exact-match answers."""
    qs = [dict(type="mcq", question="unit-test mcq", options=["a", "b"], answer="b", key_concepts=["k1"], explanation=""),
          dict(type="tf", question="unit-test tf", options=["True", "False"], answer="True", key_concepts=["k2"], explanation=""),
          dict(type="fill", question="unit-test fill ____", options=[], answer="mitochondria", key_concepts=["k3"], explanation="")]
    ev = evaluate(qs, ["b", "True", "Mitochondria"])
    ok1 = ev["total"] == 3 and all(r["verdict"] == "correct" for r in ev["results"])
    ev2 = evaluate(qs, ["", "", ""])
    ok2 = ev2["total"] == 0 and all(r["verdict"] == "unanswered" for r in ev2["results"])
    show_html(callout("ok" if ok1 and ok2 else "bad", f"Objective grading paths: all-correct → {'✅' if ok1 else '❌'}, blank answers → {'✅' if ok2 else '❌'}. "
                      "(Wrong answers and free-text answers are graded by the AI against the expected answer + source quote — test those with the widgets above.)"))
    return ok1 and ok2

def run_validation_tests():
    ok = _doc_guard_tests()
    return grading_logic_test() and ok


run_validation_tests()